In [ ]:
# [Cell 1: Check GPU]
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
# Check GPU availability (important for Kaggle)
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("⚠️ No GPU detected. Training will be slower on CPU.")

In [2]:
# [Cell 2: Clone Repo & Install Requirements]
import os
REPO_URL = "https://github.com/ariellembong/road-damage-detection-yolo.git"
REPO_DIR = "/kaggle/working/road-damage-detection-yolo"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!pip install -q ultralytics wandb

Cloning into '/kaggle/working/road-damage-detection-yolo'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 36 (delta 13), reused 32 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 5.23 MiB | 4.50 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/kaggle/working/road-damage-detection-yolo
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 601.7 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
# [Cell 3: Verify Dataset Path & Generate Working data.yaml]
import yaml
from pathlib import Path

INPUT_DATA_YAML = Path("/kaggle/input/datasets/larielembong/rdd-yolo-dataset-processed/road-damage-detection-yolo/data/data.yaml")
assert INPUT_DATA_YAML.exists(), f"❌ {INPUT_DATA_YAML} not found! Check attached dataset name."

# Read original data.yaml and point 'path' to the mounted /kaggle/input directory
with open(INPUT_DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

DATASET_ROOT = INPUT_DATA_YAML.parent
data_cfg['path'] = str(DATASET_ROOT)

# Save corrected data.yaml in /kaggle/working/ (writable)
DATA_YAML = Path('/kaggle/working/data.yaml')
with open(DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

# Verify validation images exist
val_img_dir = DATASET_ROOT / data_cfg.get('val', 'val/images')
assert val_img_dir.exists(), f"❌ Validation images not found at: {val_img_dir}"
val_count = len(list(val_img_dir.glob('*')))

print(f"✅ Fixed data.yaml created at: {DATA_YAML}")
print(f"📂 Dataset Root: {DATASET_ROOT}")
print(f"🖼️ Verified {val_count} validation images ready for training!")

In [5]:
# [Cell 4: Run Training]
# Hyperparameters are loaded automatically from configs/hyp.yaml!
!python train.py \
    --model yolo12n.pt \
    --data {DATA_YAML} \
    --name acc2_yolo12n \
    --cfg configs/hyp.yaml

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ℹ️ W&B API Key not found in secrets. Running offline or with cached login.
🚀 Starting Run: acc2_yolo12n
Model: yolo12n.pt
Data: /kaggle/input/datasets/larielembong/rdd-yolo-dataset-processed/road-damage-detection-yolo/data/data.yaml
Config: configs/hyp.yaml
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cpu 
Traceback (most recent call last):
  File "/kaggle/working/road-damage-detection-yolo/train.py", line 116, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/road-damage-detection-yolo/train.py", line 98, in main
    results = model.train(**train_kwargs)
  File "/usr/local/lib/python3.13/dist-packages/ultralytics/engine/model.py", line 885, in train
    self.trainer = (trainer or self

In [ ]:
# [Cell 5: Confirm Zipped Downloadable Archive]
!ls -lh /kaggle/working/*.zip